In [1]:
import geopandas as gpd

In [30]:
import requests, json, glob, os
import geopandas as gpd
import pandas as pd

API = "http://localhost:8000/disturbed_pairs"
OWN_TMPL = "../chapter-1/data/rq3/gedi_pairs_fs_{i}_400_with_pre.parquet"
key = ["shot_number", "new_shot_num_1"]

def burn_date_of(gj):
    """Pull burn_date from the first feature's properties (or top-level)."""
    if gj.get("type") == "FeatureCollection":
        props = gj["features"][0].get("properties", {})
    elif gj.get("type") == "Feature":
        props = gj.get("properties", {})
    else:
        props = {}
    bd = props.get("burn_date")
    if bd is None:
        raise KeyError("no burn_date in geojson properties")
    return str(pd.to_datetime(bd).date())          # normalize to YYYY-MM-DD

results = []
for i in range(5):
    path = f"test_{i}.geojson"
    if not os.path.exists(path):
        print(f"[{i}] MISSING {path}"); continue
    gj = json.load(open(path))
    fire = burn_date_of(gj)

    # --- call the API ---
    payload = {"geojson": gj, "disturbance_date": fire}
    try:
        r = requests.post(API, json=payload, timeout=1800)   # EE extraction is slow
        r.raise_for_status()
        api_fc = r.json()
    except Exception as e:
        print(f"[{i}] API error: {e}"); continue

    feats = api_fc.get("features", [])
    if not feats:
        print(f"[{i}] fire={fire}  API returned 0 pairs"); continue
    api_gdf = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")

    # --- load matching own_gdf ---
    own_path = OWN_TMPL.format(i=i)
    if not os.path.exists(own_path):
        print(f"[{i}] MISSING own file {own_path}"); continue
    own_gdf = gpd.read_parquet(own_path)

    # --- compare (your logic, dtype-safe) ---
    if not set(key).issubset(api_gdf.columns) or not set(key).issubset(own_gdf.columns):
        print(f"[{i}] missing key cols — api {list(api_gdf.columns)[:6]}… own {list(own_gdf.columns)[:6]}…")
        continue
    a = api_gdf[key].astype(str)
    b = own_gdf[key].astype(str)
    merged = b.merge(a.drop_duplicates(), on=key, how="left", indicator=True)
    n_match = (merged["_merge"] == "both").sum()

    results.append({"test": i, "fire": fire, "own": len(own_gdf), "api": len(api_gdf),
                    "matched": int(n_match), "own_not_in_api": len(own_gdf) - int(n_match),
                    "pct": round(100 * n_match / max(len(own_gdf), 1), 1)})
    print(f"[{i}] fire={fire}  own={len(own_gdf)}  api={len(api_gdf)}  "
          f"matched={n_match} ({results[-1]['pct']}%)")

summary = pd.DataFrame(results)
print("\n", summary.to_string(index=False) if len(summary) else "no successful comparisons")
if len(summary):
    print(f"\nmean match: {summary['pct'].mean():.1f}%")

[0] fire=2022-08-17  own=6335  api=6668  matched=6096 (96.2%)
[1] fire=2022-07-15  own=1415  api=1557  matched=1387 (98.0%)
[2] fire=2022-08-20  own=1657  api=1759  matched=1577 (95.2%)
[3] fire=2022-07-17  own=656  api=744  matched=628 (95.7%)
[4] fire=2022-06-18  own=250  api=322  matched=244 (97.6%)

  test       fire  own  api  matched  own_not_in_api  pct
    0 2022-08-17 6335 6668     6096             239 96.2
    1 2022-07-15 1415 1557     1387              28 98.0
    2 2022-08-20 1657 1759     1577              80 95.2
    3 2022-07-17  656  744      628              28 95.7
    4 2022-06-18  250  322      244               6 97.6

mean match: 96.5%


In [29]:
api_gdf = gpd.read_file('/Users/mdomind/Downloads/disturbed_pairs (20).geojson')

own_gdf = gpd.read_parquet('../chapter-1/data/rq3/gedi_pairs_fs_1_400_with_pre.parquet')

key = ["shot_number", "new_shot_num_1"]

a = api_gdf[key].astype(str)
b = own_gdf[key].astype(str)

# count own_gdf rows whose (shot_number, new_shot_num_1) also appears in api_gdf
merged = b.merge(a.drop_duplicates(), on=key, how="left", indicator=True)
n_match = (merged["_merge"] == "both").sum()

print(f"own_gdf pairs:            {len(own_gdf)}")
print(f"api_gdf pairs:            {len(api_gdf)}")
print(f"own pairs also in api:    {n_match}")
print(f"own pairs NOT in api:     {len(own_gdf) - n_match}")

own_gdf pairs:            1415
api_gdf pairs:            1557
own pairs also in api:    1385
own pairs NOT in api:     30


In [27]:
key = ["shot_number", "new_shot_num_1"]

a = api_gdf[key].astype(str)
b = own_gdf[key].astype(str)

# count own_gdf rows whose (shot_number, new_shot_num_1) also appears in api_gdf
merged = b.merge(a.drop_duplicates(), on=key, how="left", indicator=True)
n_match = (merged["_merge"] == "both").sum()

print(f"own_gdf pairs:            {len(own_gdf)}")
print(f"api_gdf pairs:            {len(api_gdf)}")
print(f"own pairs also in api:    {n_match}")
print(f"own pairs NOT in api:     {len(own_gdf) - n_match}")

own_gdf pairs:            1415
api_gdf pairs:            1557
own pairs also in api:    1385
own pairs NOT in api:     30


In [28]:
import ast, json, numpy as np, pandas as pd

def parse_feats(x):
    """string-dict or dict -> flat {feature: value}. Handles {'VV':{0:..}} and {'VV':..}."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, str):
        try: x = json.loads(x)
        except Exception:
            try: x = ast.literal_eval(x)
            except Exception: return None
    if not isinstance(x, dict):
        return None
    return {k: (list(v.values())[0] if isinstance(v, dict) else v) for k, v in x.items()}

key = ["shot_number", "new_shot_num_1"]
a = api_gdf.copy(); b = own_gdf.copy()
for df in (a, b):
    df["shot_number"]     = df["shot_number"].astype(str)
    df["new_shot_num_1"]  = df["new_shot_num_1"].astype(str)

# pairs present in BOTH (same query AND same match)
common = a.merge(b, on=key, how="inner", suffixes=("_api", "_own"))
print(f"api {len(a)} | own {len(b)} | coinciding pairs {len(common)}")

# per-pair, per-feature difference for s2_feats (query) and s1_feats (match)
rows = []
for _, r in common.iterrows():
    for which in ("s2_feats", "s1_feats"):
        fa, fo = parse_feats(r[f"{which}_api"]), parse_feats(r[f"{which}_own"])
        if not fa or not fo:
            continue
        for f in set(fa) & set(fo):
            try:
                rows.append({"pair": r["shot_number"], "which": which, "feature": f,
                             "api": float(fa[f]), "own": float(fo[f]),
                             "diff": float(fa[f]) - float(fo[f])})
            except (TypeError, ValueError):
                pass
cmp = pd.DataFrame(rows)

# summary: mean/abs difference per feature, split by s1 (match) vs s2 (query)
summary = (cmp.groupby(["which", "feature"])
              .agg(mean_diff=("diff", "mean"),
                   mean_abs=("diff", lambda s: s.abs().mean()),
                   max_abs=("diff", lambda s: s.abs().max()),
                   n=("diff", "size"))
              .round(4).sort_values(["which", "mean_abs"], ascending=[True, False]))
print(summary.to_string())

# quick verdict
print(f"\noverall mean |diff|: {cmp['diff'].abs().mean():.4f}")
print(f"features with mean |diff| > 0.05:\n",
      summary[summary['mean_abs'] > 0.05].index.tolist())

api 1557 | own 1415 | coinciding pairs 1385
                      mean_diff  mean_abs  max_abs     n
which    feature                                        
s1_feats B2              0.0067    0.0240   0.6747  1385
         B1              0.0004    0.0216   0.4010  1385
         B4              0.0043    0.0205   0.3590  1385
         B3              0.0047    0.0199   0.3421  1385
         B5              0.0064    0.0162   0.3291  1385
         B11             0.0059    0.0139   0.1482  1385
         B12             0.0035    0.0127   0.2347  1385
         VV              0.0026    0.0079   0.1066  1385
         B6              0.0052    0.0077   0.2123  1385
         VH              0.0061    0.0070   0.0755  1385
         B7              0.0042    0.0062   0.1956  1385
         X              -0.0039    0.0061   1.4610  1385
         B8              0.0024    0.0050   0.2108  1385
         B9              0.0032    0.0046   0.1261  1385
         Y               0.0026    0.0034   

In [8]:
import ast
import json
import numpy as np

def parse_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return set()
    
    # Extract native collection if string
    if isinstance(x, str):
        try:
            x = json.loads(x)
        except Exception:
            try:
                x = ast.literal_eval(x)
            except Exception:
                return set()

    # Convert all items to standard Python integers for uniform typing
    return {int(item) for item in x}

# Apply normalized parsing to both columns
common["cand_api"] = common["possible_pairs_api"].map(parse_list)
common["cand_own"] = common["possible_pairs_own"].map(parse_list)

# Metrics evaluation
common["cand_same"] = common.apply(lambda r: r["cand_api"] == r["cand_own"], axis=1)
common["jaccard"]   = common.apply(
    lambda r: len(r["cand_api"] & r["cand_own"]) / max(len(r["cand_api"] | r["cand_own"]), 1), 
    axis=1
)

print(f"identical candidate sets: {common['cand_same'].mean()*100:.0f}%")
print(f"mean Jaccard overlap:     {common['jaccard'].mean():.3f}")

identical candidate sets: 97%
mean Jaccard overlap:     0.997


In [12]:
common[~common['cand_same']]

,id,latitude_api,longitude_api,time_api,shot_number,rh_p98,agbd_api,disturbed,dist_api,shot_num_2_api,...,change_rh75,noise_rh75,raster_rh98_2,raster_rh98_1,change_rh98,noise_rh98,cand_api,cand_own,cand_same,jaccard
9,44,41.915815,0.981530,2022-09-03,210970800300230454,9.06,27.264793,True,2022-07-01,210970800300230460,...,2.72,1.6875,9.6202,11.4792,2.620000,1.8590,"{41670100300226058, 41670100300226059, 4167010...","{41670200300230277, 41670200300230278, 4167020...",False,0.904762
10,45,41.916114,0.980966,2022-09-03,210970800300230453,3.76,3.406508,True,2022-07-01,210970800300230460,...,-1.12,4.6300,4.1756,11.4792,-2.600000,7.3036,"{41670100300226057, 41670100300226058, 4167010...","{41670100300226057, 41670100300226058, 4167010...",False,0.918605
11,57,41.923695,0.996805,2022-09-03,210970500300227028,5.93,10.485086,True,2022-07-01,210970500300227040,...,0.22,-2.4200,6.4600,0.1600,2.570000,-6.3000,"{73250500300394758, 73250500300394759, 7325050...","{73250500300394758, 73250500300394759, 7325050...",False,0.914894
13,63,41.942726,0.991327,2022-09-03,210970200300231219,12.69,75.230881,True,2022-07-01,210970200300231230,...,-1.71,-1.0000,3.2300,4.7164,0.220000,1.4864,"{68370800300394609, 68370800300394610, 6837080...","{88770200200238848, 88770200200238849, 8877020...",False,0.925000
14,64,41.943028,0.990755,2022-09-03,210970200300231218,12.98,60.519337,True,2022-07-01,210970200300231230,...,3.47,-0.3525,4.0808,3.7064,7.639999,-0.3744,"{68370800300394608, 68370800300394609, 6837080...","{88770200200238848, 88770200200238849, 8877020...",False,0.937500
15,65,41.938912,0.998579,2022-09-03,210970200300231232,8.09,42.175026,True,2022-07-01,210970200300231230,...,1.38,-3.6900,9.8992,6.4348,0.320000,-3.4644,"{68370800300394624, 68370800300394625, 6837080...","{68370800300394624, 68370800300394625, 6837080...",False,0.928571
